# TabR feature pruning notebook

Notebook baru untuk eksperimen **feature pruning** pada setup final TabR.

Prinsip:
- **tidak replace** hasil run lama
- dataset/exp baru pakai suffix `*_prune_*`
- default `RESET_PRUNING_OUTPUTS = False`
- split train/val/external test mengikuti notebook sebelumnya
- pruning dilakukan pada **fitur asli**, bukan PCA/LDA/4QMV


In [ ]:
from __future__ import annotations

import copy
import json
import os
import random
import shutil
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display
from scipy.special import expit
from sklearn.feature_selection import mutual_info_classif
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorboard.backend.event_processing import event_accumulator

sns.set_theme(style='whitegrid')
ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

TABR_ROOT = ROOT / 'third_party' / 'tabular-dl-tabr-official'
os.environ['PROJECT_DIR'] = str(TABR_ROOT)
if str(TABR_ROOT) not in sys.path:
    sys.path.append(str(TABR_ROOT))

import lib
import bin.go

FEATURES_PATH = ROOT / 'output' / 'apex' / 'features' / 'poc_abs_flatten_ordered.xlsx'
BASE_TUNING_TOML = TABR_ROOT / 'exp' / 'tabr' / 'convat_apex_anxiety' / '0-tuning.toml'
DATA_ROOT = TABR_ROOT / 'data'
EXP_ROOT = TABR_ROOT / 'exp' / 'tabr'

RANDOM_SEED = 72
N_EXTERNAL_PER_LABEL = 20
N_EVAL_SEEDS = 15
N_ENSEMBLES = 3
ENSEMBLE_SIZE = 5
RESET_PRUNING_OUTPUTS = False
CUDA_VISIBLE_DEVICES = os.environ.get('CUDA_VISIBLE_DEVICES', '0')
os.environ['CUDA_VISIBLE_DEVICES'] = CUDA_VISIBLE_DEVICES

LABEL_MAP = {'anxiety_rendah': 0, 'anxiety_tinggi': 1}
LABEL_NAMES = ['anxiety_rendah', 'anxiety_tinggi']
META_COLS = [
    'phase', 'condition', 'label', 'participant', 'participant_raw', 'question', 'question_no',
    'sample', 'clip', 'event_clip', 'event_no', 'clip_path', 'frame', 'target', 'event_id',
]

EXPERIMENTS = [
    {
        'mode': 'variance_corr',
        'dataset_name': 'convat_apex_anxiety_prune_variance_corr',
        'exp_name': 'convat_apex_anxiety_prune_variance_corr',
        'variance_threshold': 1e-8,
        'corr_threshold': 0.95,
    },
    {
        'mode': 'mi_topk',
        'dataset_name': 'convat_apex_anxiety_prune_mi128',
        'exp_name': 'convat_apex_anxiety_prune_mi128',
        'top_k': 128,
    },
    {
        'mode': 'mi_topk',
        'dataset_name': 'convat_apex_anxiety_prune_mi256',
        'exp_name': 'convat_apex_anxiety_prune_mi256',
        'top_k': 256,
    },
]

display(Markdown(
    f'- ROOT: `{ROOT}`\n'
    f'- FEATURES: `{FEATURES_PATH}`\n'
    f'- BASE_TUNING_TOML: `{BASE_TUNING_TOML}`\n'
    f'- RESET_PRUNING_OUTPUTS: `{RESET_PRUNING_OUTPUTS}`'
))


In [ ]:
if not FEATURES_PATH.exists():
    raise FileNotFoundError(FEATURES_PATH)
if not BASE_TUNING_TOML.exists():
    raise FileNotFoundError(BASE_TUNING_TOML)

df_raw = pd.read_excel(FEATURES_PATH)
df_raw = df_raw[df_raw['label'].isin(LABEL_MAP)].copy()
df_raw['target'] = df_raw['label'].map(LABEL_MAP).astype(int)
df_raw['event_id'] = (
    df_raw['phase'].astype(str) + '||'
    + df_raw['participant'].astype(str) + '||'
    + df_raw['question'].astype(str) + '||'
    + df_raw['clip'].astype(str) + '||'
    + df_raw['event_clip'].astype(str)
)
BASE_FEATURE_COLS = [c for c in df_raw.columns if c not in META_COLS[:-2]]
BASE_FEATURE_COLS = [c for c in BASE_FEATURE_COLS if c not in {'target', 'event_id'}]

print('rows:', len(df_raw))
print('feature cols:', len(BASE_FEATURE_COLS))
print(df_raw['label'].value_counts().to_dict())


In [ ]:
def balanced_external_events(event_table: pd.DataFrame, n_per_label: int, seed: int = RANDOM_SEED) -> set[str]:
    rng = random.Random(seed)
    selected_ids: list[str] = []
    for label_name in LABEL_NAMES:
        label_df = event_table[event_table['label'] == label_name].copy()
        picked_rows = []
        used_ids = set()
        participant_groups = []
        for participant, part_df in label_df.groupby('participant', sort=True):
            part_df = part_df.sort_values(['phase', 'question', 'clip', 'event_clip'], kind='stable')
            phase_groups = []
            for _phase, phase_df in part_df.groupby('phase', sort=True):
                phase_groups.append(phase_df.to_dict('records'))
            participant_groups.append((participant, phase_groups))
        while len(picked_rows) < n_per_label:
            progress = False
            for _participant, phase_groups in participant_groups:
                for records in phase_groups:
                    while records and records[0]['event_id'] in used_ids:
                        records.pop(0)
                    if not records:
                        continue
                    row = records.pop(0)
                    picked_rows.append(row)
                    used_ids.add(row['event_id'])
                    progress = True
                    if len(picked_rows) >= n_per_label:
                        break
                if len(picked_rows) >= n_per_label:
                    break
            if not progress:
                break
        if len(picked_rows) < n_per_label:
            remaining = label_df[~label_df['event_id'].isin(used_ids)].sort_values(
                ['participant', 'phase', 'question', 'clip', 'event_clip'], kind='stable'
            )
            for row in remaining.to_dict('records'):
                picked_rows.append(row)
                used_ids.add(row['event_id'])
                if len(picked_rows) >= n_per_label:
                    break
        selected_ids.extend([row['event_id'] for row in picked_rows[:n_per_label]])
    return set(selected_ids)


def balance_train_clips_by_label(df_train: pd.DataFrame, seed: int = RANDOM_SEED) -> pd.DataFrame:
    rng = random.Random(seed)
    event_table_train = (
        df_train[['event_id', 'label', 'target', 'phase', 'participant', 'question', 'clip', 'event_clip']]
        .drop_duplicates()
        .reset_index(drop=True)
    )
    grouped = {}
    for label_name, label_df in event_table_train.groupby('label', sort=True):
        clip_table = (
            label_df[['participant', 'question', 'clip', 'label']]
            .drop_duplicates()
            .sort_values(['participant', 'question', 'clip'], kind='stable')
            .reset_index(drop=True)
        )
        grouped[label_name] = clip_table
    min_clip_count = min(len(x) for x in grouped.values())
    selected_clip_keys = []
    for _label_name, clip_table in grouped.items():
        clip_records = clip_table.to_dict('records')
        rng.shuffle(clip_records)
        chosen = clip_records[:min_clip_count]
        selected_clip_keys.extend([
            (row['participant'], row['question'], row['clip'], row['label'])
            for row in chosen
        ])
    selected_clip_keys = set(selected_clip_keys)
    return df_train[
        df_train.apply(
            lambda row: (row['participant'], row['question'], row['clip'], row['label']) in selected_clip_keys,
            axis=1,
        )
    ].copy()


event_table = df_raw[['event_id', 'label', 'target', 'phase', 'participant', 'question', 'clip', 'event_clip']].drop_duplicates().reset_index(drop=True)
external_event_ids = balanced_external_events(event_table, N_EXTERNAL_PER_LABEL, seed=RANDOM_SEED)
df_external_base = df_raw[df_raw['event_id'].isin(external_event_ids)].copy()
df_train_all_base = df_raw[~df_raw['event_id'].isin(external_event_ids)].copy()
df_train_balanced_base = balance_train_clips_by_label(df_train_all_base, seed=RANDOM_SEED)
train_event_table_balanced = df_train_balanced_base[['event_id', 'label', 'target']].drop_duplicates().reset_index(drop=True)
train_event_ids, val_event_ids = train_test_split(
    train_event_table_balanced['event_id'],
    test_size=0.3,
    stratify=train_event_table_balanced['target'],
    random_state=RANDOM_SEED,
)
train_event_ids = set(train_event_ids.tolist())
val_event_ids = set(val_event_ids.tolist())

df_train_base = df_train_balanced_base[df_train_balanced_base['event_id'].isin(train_event_ids)].copy()
df_val_base = df_train_balanced_base[df_train_balanced_base['event_id'].isin(val_event_ids)].copy()

print('train rows:', len(df_train_base))
print('val rows:', len(df_val_base))
print('external rows:', len(df_external_base))
print('train events:', df_train_base['event_id'].nunique())
print('val events:', df_val_base['event_id'].nunique())
print('external events:', df_external_base['event_id'].nunique())


In [ ]:
def fit_preprocessor(feature_cols: list[str]) -> tuple[SimpleImputer, StandardScaler, np.ndarray, np.ndarray, np.ndarray]:
    imputer = SimpleImputer(strategy='mean')
    scaler = StandardScaler()
    X_train = scaler.fit_transform(imputer.fit_transform(df_train_base[feature_cols]))
    X_val = scaler.transform(imputer.transform(df_val_base[feature_cols]))
    X_test = scaler.transform(imputer.transform(df_external_base[feature_cols]))
    return imputer, scaler, X_train, X_val, X_test


def select_variance_corr_features(
    feature_cols: list[str],
    variance_threshold: float = 1e-8,
    corr_threshold: float = 0.95,
) -> tuple[list[str], dict]:
    _imputer, _scaler, X_train, _X_val, _X_test = fit_preprocessor(feature_cols)
    variances = pd.Series(np.var(X_train, axis=0), index=feature_cols)
    kept = variances[variances > variance_threshold].index.tolist()
    X_kept = pd.DataFrame(X_train[:, [feature_cols.index(c) for c in kept]], columns=kept)
    corr = X_kept.corr().abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    to_drop = [column for column in upper.columns if (upper[column] > corr_threshold).any()]
    selected = [c for c in kept if c not in to_drop]
    meta = {
        'method': 'variance_corr',
        'variance_threshold': variance_threshold,
        'corr_threshold': corr_threshold,
        'n_input_features': len(feature_cols),
        'n_after_variance': len(kept),
        'n_dropped_corr': len(to_drop),
        'n_selected_features': len(selected),
    }
    return selected, meta


def select_mi_topk_features(feature_cols: list[str], top_k: int) -> tuple[list[str], dict]:
    _imputer, _scaler, X_train, _X_val, _X_test = fit_preprocessor(feature_cols)
    y_train = df_train_base['target'].to_numpy(dtype=np.int64)
    mi = mutual_info_classif(X_train, y_train, random_state=RANDOM_SEED)
    mi_series = pd.Series(mi, index=feature_cols).sort_values(ascending=False)
    top_k = min(top_k, len(mi_series))
    selected = mi_series.head(top_k).index.tolist()
    meta = {
        'method': 'mi_topk',
        'top_k': top_k,
        'n_input_features': len(feature_cols),
        'n_selected_features': len(selected),
        'mi_min_selected': float(mi_series.head(top_k).iloc[-1]),
        'mi_max': float(mi_series.iloc[0]),
    }
    return selected, meta


def build_pruned_dataframe(config: dict) -> tuple[pd.DataFrame, list[str], dict]:
    mode = config['mode']
    if mode == 'variance_corr':
        selected, meta = select_variance_corr_features(
            BASE_FEATURE_COLS,
            variance_threshold=config['variance_threshold'],
            corr_threshold=config['corr_threshold'],
        )
    elif mode == 'mi_topk':
        selected, meta = select_mi_topk_features(BASE_FEATURE_COLS, top_k=config['top_k'])
    else:
        raise ValueError(f'Unknown pruning mode: {mode}')

    df_mode = df_raw[META_COLS + selected].copy()
    meta['selected_feature_cols'] = selected
    return df_mode, selected, meta


In [ ]:
def export_official_dataset(df_mode: pd.DataFrame, feature_cols: list[str], dataset_name: str, reset_existing: bool = RESET_PRUNING_OUTPUTS) -> dict:
    out_dir = DATA_ROOT / dataset_name
    if out_dir.exists():
        if reset_existing:
            shutil.rmtree(out_dir)
        else:
            raise FileExistsError(f'Dataset dir already exists: {out_dir}')

    df_tr = df_mode[df_mode['event_id'].isin(train_event_ids)].copy()
    df_val = df_mode[df_mode['event_id'].isin(val_event_ids)].copy()
    df_test = df_mode[df_mode['event_id'].isin(external_event_ids)].copy()

    imputer = SimpleImputer(strategy='mean')
    scaler = StandardScaler()
    X_train = scaler.fit_transform(imputer.fit_transform(df_tr[feature_cols])).astype(np.float32)
    X_val = scaler.transform(imputer.transform(df_val[feature_cols])).astype(np.float32)
    X_test = scaler.transform(imputer.transform(df_test[feature_cols])).astype(np.float32)
    y_train = df_tr['target'].to_numpy(dtype=np.int64)
    y_val = df_val['target'].to_numpy(dtype=np.int64)
    y_test = df_test['target'].to_numpy(dtype=np.int64)

    out_dir.mkdir(parents=True, exist_ok=False)
    np.save(out_dir / 'X_num_train.npy', X_train)
    np.save(out_dir / 'X_num_val.npy', X_val)
    np.save(out_dir / 'X_num_test.npy', X_test)
    np.save(out_dir / 'Y_train.npy', y_train)
    np.save(out_dir / 'Y_val.npy', y_val)
    np.save(out_dir / 'Y_test.npy', y_test)
    (out_dir / 'READY').write_text('')
    (out_dir / 'info.json').write_text(json.dumps({'task_type': 'binclass', 'name': dataset_name, 'id': dataset_name}, indent=2))
    (out_dir / 'feature_cols.json').write_text(json.dumps(feature_cols, indent=2))
    df_tr.to_csv(out_dir / 'train_split.csv', index=False)
    df_val.to_csv(out_dir / 'val_split.csv', index=False)
    df_test.to_csv(out_dir / 'test_split.csv', index=False)
    return {
        'dataset_dir': out_dir,
        'df_train': df_tr,
        'df_val': df_val,
        'df_test': df_test,
        'X_train': X_train,
        'X_val': X_val,
        'X_test': X_test,
        'y_train': y_train,
        'y_val': y_val,
        'y_test': y_test,
    }


def prepare_tuning_config(dataset_name: str, exp_name: str, reset_existing: bool = RESET_PRUNING_OUTPUTS) -> tuple[dict, Path]:
    config = lib.load_config(BASE_TUNING_TOML)
    config = copy.deepcopy(config)
    config['space']['data']['path'] = f':data/{dataset_name}'
    tuning_dir = EXP_ROOT / exp_name / '0-tuning'
    tuning_toml = tuning_dir.with_suffix('.toml')
    exp_dir = EXP_ROOT / exp_name
    if exp_dir.exists():
        if reset_existing:
            shutil.rmtree(exp_dir)
        else:
            raise FileExistsError(f'Experiment path already exists: {exp_dir}')
    elif tuning_toml.exists():
        if reset_existing:
            tuning_toml.unlink()
        else:
            raise FileExistsError(f'Experiment config already exists: {tuning_toml}')
    tuning_toml.parent.mkdir(parents=True, exist_ok=True)
    lib.dump_config(config, tuning_toml)
    return config, tuning_toml


In [ ]:
def load_scalar_series(run_dir: Path, tag_dir: str, tag: str) -> pd.DataFrame:
    event_file = next((run_dir / tag_dir).glob('events.out.tfevents.*'))
    ea = event_accumulator.EventAccumulator(str(event_file))
    ea.Reload()
    scalars = ea.Scalars(tag)
    return pd.DataFrame({
        'epoch': [x.step for x in scalars],
        'value': [x.value for x in scalars],
        'wall_time': [x.wall_time for x in scalars],
    })


def collect_epoch_metrics(run_dir: Path) -> pd.DataFrame:
    checkpoint = lib.load_checkpoint(run_dir, map_location='cpu')
    rows = []
    for item in checkpoint['training_log']:
        metrics = item['metrics']
        rows.append({
            'epoch': len(rows),
            'val_loss': float(metrics['val'].get('cross-entropy', np.nan)),
            'test_loss': float(metrics['test'].get('cross-entropy', np.nan)),
            'val_acc': float(metrics['val']['score']),
            'test_acc': float(metrics['test']['score']),
        })
    epoch_df = pd.DataFrame(rows)
    train_loss = load_scalar_series(run_dir, 'loss_train', 'loss')
    epoch_df = epoch_df.merge(train_loss[['epoch', 'value']].rename(columns={'value': 'train_loss'}), on='epoch', how='left')
    return epoch_df


def infer_train_accuracy_from_predictions(run_dir: Path, y_train: np.ndarray) -> float:
    preds = lib.load_predictions(run_dir)
    train_logits = preds['train']
    train_prob = expit(train_logits)
    train_pred = (train_prob >= 0.5).astype(int)
    return float((train_pred == y_train).mean())


def make_confusion_df(y_true: np.ndarray, y_pred: np.ndarray) -> pd.DataFrame:
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    return pd.DataFrame(cm, index=['true_anxiety_rendah', 'true_anxiety_tinggi'], columns=['pred_anxiety_rendah', 'pred_anxiety_tinggi'])


def evaluate_best_run(run_dir: Path, dataset_name: str, split_info: dict) -> dict:
    dataset = lib.build_dataset(path=f':data/{dataset_name}', cache=True, num_policy=None, cat_policy=None, y_policy=None, seed=RANDOM_SEED)
    preds = lib.load_predictions(run_dir)
    out = {}
    for part, y_true in [('train', split_info['y_train']), ('val', split_info['y_val']), ('test', split_info['y_test'])]:
        logits = preds[part]
        prob = expit(logits)
        pred = (prob >= 0.5).astype(int)
        out[part] = {
            'y_true': y_true,
            'prob': prob,
            'pred': pred,
            'report': classification_report(y_true, pred, target_names=LABEL_NAMES, output_dict=True),
            'cm_df': make_confusion_df(y_true, pred),
        }
    out['dataset_name'] = dataset_name
    out['dataset'] = dataset
    out['report_json'] = json.loads((run_dir / 'report.json').read_text())
    out['summary_json'] = json.loads((run_dir / 'summary.json').read_text())
    out['epoch_df'] = collect_epoch_metrics(run_dir)
    out['epoch_df']['train_acc'] = np.nan
    out['epoch_df']['best_epoch'] = int(out['report_json']['best_epoch'])
    out['final_train_acc'] = infer_train_accuracy_from_predictions(run_dir, split_info['y_train'])
    return out


def summarize_result(mode: str, dataset_name: str, exp_name: str, result: dict, transform_meta: dict) -> pd.DataFrame:
    rows = []
    for part, part_name in [('train', 'train'), ('val', 'val'), ('test', 'external_test')]:
        report = result[part]['report']
        rows.append({
            'mode': mode,
            'dataset_name': dataset_name,
            'exp_name': exp_name,
            'split': part_name,
            'accuracy': report['accuracy'],
            'macro_f1': report['macro avg']['f1-score'],
            'weighted_f1': report['weighted avg']['f1-score'],
            'precision_tinggi': report['anxiety_tinggi']['precision'],
            'recall_tinggi': report['anxiety_tinggi']['recall'],
            'n_features': transform_meta.get('n_selected_features'),
            'best_epoch': result['report_json']['best_epoch'],
        })
    return pd.DataFrame(rows)


## Jalankan pruning experiment

Cell di bawah:
1. pilih fitur hasil pruning
2. export dataset baru
3. tulis config tuning baru
4. jalankan `bin.go.main(...)`

Karena `RESET_PRUNING_OUTPUTS = False`, nama dataset/exp yang sudah ada **akan error**, bukan tertimpa.


In [ ]:
prepared_runs = []
for cfg in EXPERIMENTS:
    df_mode, feature_cols_mode, transform_meta = build_pruned_dataframe(cfg)
    split_info = export_official_dataset(df_mode, feature_cols_mode, cfg['dataset_name'])
    _config, tuning_toml = prepare_tuning_config(cfg['dataset_name'], cfg['exp_name'])
    prepared_runs.append({
        'cfg': cfg,
        'feature_cols_mode': feature_cols_mode,
        'transform_meta': transform_meta,
        'split_info': split_info,
        'tuning_toml': tuning_toml,
    })
    display(Markdown(
        f"Prepared `{cfg['dataset_name']}` / `{cfg['exp_name']}` with `{len(feature_cols_mode)}` features.\n\n"
        f"- tuning: `{tuning_toml}`"
    ))


In [ ]:
# Jalankan kalau sudah siap. Tidak replace output lama.
for item in prepared_runs:
    print('RUN:', item['cfg']['exp_name'])
    bin.go.main(
        item['tuning_toml'],
        n_seeds=N_EVAL_SEEDS,
        n_ensembles=N_ENSEMBLES,
        ensemble_size=ENSEMBLE_SIZE,
        force=False,
    )


## Ringkas hasil single best seed

Cell ini baca hasil dari `0-evaluation`, lalu pilih seed terbaik berdasarkan `val.score`.


In [ ]:
all_summaries = []
all_run_results = {}

for item in prepared_runs:
    cfg = item['cfg']
    transform_meta = item['transform_meta']
    split_info = item['split_info']
    eval_dir = EXP_ROOT / cfg['exp_name'] / '0-evaluation'
    best_seed = max(
        range(N_EVAL_SEEDS),
        key=lambda s: json.loads((eval_dir / str(s) / 'report.json').read_text())['metrics']['val']['score'],
    )
    best_run_dir = eval_dir / str(best_seed)
    result = evaluate_best_run(best_run_dir, cfg['dataset_name'], split_info)
    summary = summarize_result(cfg['mode'], cfg['dataset_name'], cfg['exp_name'], result, transform_meta)
    summary['best_seed'] = best_seed
    all_summaries.append(summary)
    all_run_results[cfg['exp_name']] = {
        'best_seed': best_seed,
        'best_run_dir': best_run_dir,
        'result': result,
        'transform_meta': transform_meta,
    }

compare_df = pd.concat(all_summaries, ignore_index=True)
display(compare_df.sort_values(['split', 'accuracy'], ascending=[True, False]))


## Baca ensemble result

Cell ini baca ringkasan `0-ensemble-5` kalau sudah ada.


In [ ]:
ensemble_rows = []
for item in prepared_runs:
    cfg = item['cfg']
    ensemble_root = EXP_ROOT / cfg['exp_name'] / f'0-ensemble-{ENSEMBLE_SIZE}'
    if not ensemble_root.exists():
        continue
    for ensemble_dir in sorted([x for x in ensemble_root.iterdir() if x.is_dir()]):
        report_path = ensemble_dir / 'report.json'
        if not report_path.exists():
            continue
        report = json.loads(report_path.read_text())
        ensemble_rows.append({
            'exp_name': cfg['exp_name'],
            'ensemble_id': ensemble_dir.name,
            'train_score': report['scores']['train'],
            'val_score': report['scores']['val'],
            'test_score': report['scores']['test'],
        })

ensemble_df = pd.DataFrame(ensemble_rows)
if len(ensemble_df):
    display(ensemble_df.sort_values(['test_score', 'val_score'], ascending=False))
else:
    print('Belum ada output ensemble.')
